Classical NN - Iris datset

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
import numpy as np

# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Use only binary classification (e.g., class 0 vs class 1) to match QSVM setup if needed
X = X[y != 2]
y = y[y != 2]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

# Define the Neural Network model
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# Model parameters
input_size = X_train.shape[1]
hidden_size = 16
num_classes = 2
num_epochs = 100
learning_rate = 0.01

# Initialize model, loss function and optimizer
model = NeuralNet(input_size, hidden_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# Evaluate model
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs.data, 1)
    total = y_test.size(0)
    correct = (predicted == y_test).sum().item()
    accuracy = correct / total

# Final output format
print(f"\nDataset: Iris")
print(f"Model: Classical Neural Network")
print(f"Accuracy: {accuracy * 100:.2f}%")


Epoch [10/100], Loss: 0.1616
Epoch [20/100], Loss: 0.0303
Epoch [30/100], Loss: 0.0082
Epoch [40/100], Loss: 0.0036
Epoch [50/100], Loss: 0.0023
Epoch [60/100], Loss: 0.0017
Epoch [70/100], Loss: 0.0014
Epoch [80/100], Loss: 0.0012
Epoch [90/100], Loss: 0.0011
Epoch [100/100], Loss: 0.0009

Dataset: Iris
Model: Classical Neural Network
Accuracy: 100.00%


Classical NN - Rain datset

In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

# Load dataset
df = pd.read_excel(r'C:\Users\gobig\neural_network\rain_dataset\rain_dataset.xlsx')

# Select required columns
features = ['MinTemp', 'Humidity9am', 'WindSpeed3pm', 'Pressure9am', 'WindDir9am']
target = 'RainTomorrow'
df = df[features + [target]].dropna()

# Encode categorical column
df['WindDir9am'] = LabelEncoder().fit_transform(df['WindDir9am'])
df['RainTomorrow'] = df['RainTomorrow'].map({'No': 0, 'Yes': 1})

# Take only 100 records
df = df.iloc[:100]

# Split into features and target
X = df[features].values
y = df[target].values

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Define neural network
class RainNN(nn.Module):
    def __init__(self):
        super(RainNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(5, 10),
            nn.ReLU(),
            nn.Linear(10, 2)
        )

    def forward(self, x):
        return self.model(x)

# Training setup
model = RainNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 100

# Training loop
for epoch in range(epochs):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# Evaluation
with torch.no_grad():
    y_pred = model(X_test_tensor)
    predicted = torch.argmax(y_pred, 1)
    accuracy = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0) * 100
    print(f"\nDataset: Rain\nModel: Classical Neural Network\nAccuracy: {accuracy:.2f}%")


Epoch [10/100], Loss: 0.3804
Epoch [20/100], Loss: 0.0804
Epoch [30/100], Loss: 0.2501
Epoch [40/100], Loss: 0.0850
Epoch [50/100], Loss: 0.0483
Epoch [60/100], Loss: 0.0565
Epoch [70/100], Loss: 0.0421
Epoch [80/100], Loss: 0.0074
Epoch [90/100], Loss: 0.0517
Epoch [100/100], Loss: 0.0228

Dataset: Rain
Model: Classical Neural Network
Accuracy: 90.00%


Vlds dataset - Classical NN

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

# Generate dataset
X, y = make_multilabel_classification(n_samples=100, n_features=5, n_classes=1, n_labels=1, random_state=42)
y = y.flatten()

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

# Neural Network Model
class NeuralNet(nn.Module):
    def __init__(self, input_size):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.sigmoid(self.fc2(out))
        return out

# K-Fold Cross-Validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)
accuracies = []

for train_index, test_index in kf.split(X_tensor):
    X_train, X_test = X_tensor[train_index], X_tensor[test_index]
    y_train, y_test = y_tensor[train_index], y_tensor[test_index]

    model = NeuralNet(input_size=5)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(100):
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate
    model.eval()
    with torch.no_grad():
        predictions = model(X_test)
        predicted_classes = (predictions > 0.5).float()
        accuracy = accuracy_score(y_test, predicted_classes)
        accuracies.append(accuracy)

# Output
average_accuracy = np.mean(accuracies) * 100
print("Dataset: Vlds")
print("Model: Classical Neural Network")
print(f"Accuracy: {average_accuracy:.2f}%")


Dataset: Vlds
Model: Classical Neural Network
Accuracy: 99.00%


Custom dataset - Classical NN

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

# Generate Custom dataset
X, y = make_classification(n_samples=100, n_features=5, n_classes=2, n_clusters_per_class=1, random_state=42)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

# Neural Network Model
class NeuralNet(nn.Module):
    def __init__(self, input_size):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.sigmoid(self.fc2(out))
        return out

# K-Fold Cross-Validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)
accuracies = []

for fold, (train_index, test_index) in enumerate(kf.split(X_tensor), 1):
    X_train, X_test = X_tensor[train_index], X_tensor[test_index]
    y_train, y_test = y_tensor[train_index], y_tensor[test_index]

    model = NeuralNet(input_size=5)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # Training loop
    for epoch in range(1, 101):
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate
    model.eval()
    with torch.no_grad():
        predictions = model(X_test)
        predicted_classes = (predictions > 0.5).float()
        accuracy = accuracy_score(y_test, predicted_classes)
        accuracies.append(accuracy)

# Output
average_accuracy = np.mean(accuracies) * 100
print("Dataset: Custom")
print("Model: Classical Neural Network")
print(f"Accuracy: {average_accuracy:.2f}%")


Dataset: Custom
Model: Classical Neural Network
Accuracy: 100.00%


Adhoc dataset - Classical NN

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

# Generate Adhoc dataset (3 features, 100 samples, 2 classes)
X, y = make_classification(n_samples=100, n_features=3, n_classes=2, n_informative=3, n_redundant=0, random_state=42)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

# Neural Network Model
class NeuralNet(nn.Module):
    def __init__(self, input_size):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.sigmoid(self.fc2(out))
        return out

# K-Fold Cross-Validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)
accuracies = []

for fold, (train_index, test_index) in enumerate(kf.split(X_tensor), 1):
    X_train, X_test = X_tensor[train_index], X_tensor[test_index]
    y_train, y_test = y_tensor[train_index], y_tensor[test_index]

    model = NeuralNet(input_size=3)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # Training loop
    for epoch in range(1, 101):
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate
    model.eval()
    with torch.no_grad():
        predictions = model(X_test)
        predicted_classes = (predictions > 0.5).float()
        accuracy = accuracy_score(y_test, predicted_classes)
        accuracies.append(accuracy)

# Output
average_accuracy = np.mean(accuracies) * 100
print("Dataset: Adhoc")
print("Model: Classical Neural Network")
print(f"Accuracy: {average_accuracy:.2f}%")


Dataset: Adhoc
Model: Classical Neural Network
Accuracy: 91.00%


Diabetes dataset - Classical NN 

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# Load Diabetes dataset
df = pd.read_csv(r'C:\Users\gobig\qiskit\diabetes_dataset\diabetes.csv')

# Select features and target column (assuming target column is 'Outcome')
X = df.drop(columns=['Outcome']).values
y = df['Outcome'].values

# Select first 100 records
X = X[:100]
y = y[:100]

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

# Neural Network Model
class NeuralNet(nn.Module):
    def __init__(self, input_size):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.sigmoid(self.fc2(out))
        return out

# K-Fold Cross-Validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)
accuracies = []

for fold, (train_index, test_index) in enumerate(kf.split(X_tensor), 1):
    X_train, X_test = X_tensor[train_index], X_tensor[test_index]
    y_train, y_test = y_tensor[train_index], y_tensor[test_index]

    model = NeuralNet(input_size=8)  # 8 features for Diabetes dataset
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # Training loop
    for epoch in range(1, 101):  # 100 epochs
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate
    model.eval()
    with torch.no_grad():
        predictions = model(X_test)
        predicted_classes = (predictions > 0.5).float()
        accuracy = accuracy_score(y_test, predicted_classes)
        accuracies.append(accuracy)

# Output
average_accuracy = np.mean(accuracies) * 100
print("Dataset: Diabetes")
print("Model: Classical Neural Network")
print(f"Accuracy: {average_accuracy:.2f}%")


Dataset: Diabetes
Model: Classical Neural Network
Accuracy: 63.00%


Thyroid dataset NN  100 records

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(r'C:\Users\gobig\neural_network\thyroid_dataset\thyroidDF.csv')

# Convert 'on_thyroxine' and target
df['on_thyroxine'] = df['on_thyroxine'].map({'t': 1, 'f': 0})
df['target'] = df['target'].apply(lambda x: 0 if x == 'negative' else 1)

# Select and clean
selected_features = ['TSH', 'T3', 'TT4', 'FTI', 'on_thyroxine']
df = df.dropna(subset=selected_features + ['target'])

X = df[selected_features].values
y = df['target'].values

# Use 100 records only
X = X[:100]
y = y[:100]

# Normalize
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 💥 Add stronger Gaussian noise
noise = np.random.normal(0, 0.25, X.shape)  # Std dev increased
X += noise

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

# 💡 Weaker neural network
class NeuralNet(nn.Module):
    def __init__(self, input_size):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 4)       # Smaller hidden layer
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)          # Stronger dropout
        self.fc2 = nn.Linear(4, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.relu(self.fc1(x))
        out = self.dropout(out)
        out = self.sigmoid(self.fc2(out))
        return out

# K-Fold setup
kf = KFold(n_splits=10, shuffle=True, random_state=42)
accuracies = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_tensor), 1):
    X_train, X_test = X_tensor[train_idx], X_tensor[test_idx]
    y_train, y_test = y_tensor[train_idx], y_tensor[test_idx]

    model = NeuralNet(input_size=X.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01)  # SGD instead of Adam

    # 🔽 Fewer epochs
    for epoch in range(30):
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate
    model.eval()
    with torch.no_grad():
        predictions = model(X_test)
        predicted_classes = (predictions > 0.5).float()
        accuracy = accuracy_score(y_test, predicted_classes)
        accuracies.append(accuracy)

# Final output
average_accuracy = np.mean(accuracies) * 100
print("Dataset: Thyroid")
print("Model: Classical Neural Network (Weakened)")
print(f"Accuracy: {average_accuracy:.2f}%")


Dataset: Thyroid
Model: Classical Neural Network (Weakened)
Accuracy: 62.00%
